# VWAP Strategy vs Buy and Hold (Client Friendly, Parametric)

Cette strategie reconstruit un trading intraday autour de la VWAP de session. Le notebook filtre d'abord les barres RTH entre `SESSION_START` et `SESSION_END`, recalcule la `session_vwap`, l'ATR et les structures intraday, puis applique la variante selectionnee. Pour une variante discrete comme `vwap_pullback_continuation`, un long n'est pris que si le marche reste dans un regime haussier au-dessus de la VWAP, que la pente de VWAP confirme ce biais, qu'un pullback reste propre par rapport a la VWAP, puis qu'une reprise de momentum valide l'entree; le short est le miroir exact. La sortie depend ensuite du recross de VWAP si `EXIT_ON_VWAP_RECROSS=True`, du stop structurel derive de l'ATR, ou de la cloture de session.

Les parametres interviennent sur trois couches. Les parametres de structure du signal comme `ATR_PERIOD`, `ATR_BUFFER`, `COMPRESSION_LENGTH`, `PULLBACK_LOOKBACK`, `SLOPE_LOOKBACK`, `SLOPE_THRESHOLD` et `TIME_WINDOWS` rendent l'entree plus ou moins selective: plus ils sont stricts, plus on reduit le nombre de trades pour privilegier les contextes juges propres. Les parametres de sizing et d'execution comme `QUANTITY_MODE`, `FIXED_QUANTITY`, `RISK_PER_TRADE_PCT`, `TICK_SIZE`, `POINT_VALUE_USD`, `COMMISSION_PER_SIDE_USD` et `SLIPPAGE_TICKS` ne changent pas le signal, mais transforment directement ce signal en exposition, en frais et donc en PnL reel. Enfin, les garde-fous prop comme `MAX_TRADES_PER_DAY`, `MAX_LOSSES_PER_DAY`, `DAILY_STOP_THRESHOLD_USD`, `CONSECUTIVE_LOSSES_THRESHOLD` et `DELEVERAGE_AFTER_LOSING_STREAK` coupent ou reduisent l'activite quand la journee ou la sequence de pertes se degrade, ce qui change surtout le profil de risque et la regularite du run.


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while root != root.parent and not (root / "pyproject.toml").exists():
    root = root.parent

if not (root / "pyproject.toml").exists():
    raise RuntimeError("Could not locate repository root from the current working directory.")

if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")


In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import Markdown, display

from src.analytics.orb_notebook_utils import (
    build_scope_readout_markdown,
    curve_annualized_return,
    curve_daily_sharpe,
    curve_daily_vol,
    curve_max_drawdown_pct,
    curve_total_return_pct,
    format_curve_stats_line,
    normalize_curve,
)
from src.analytics.vwap_metrics import compute_extended_vwap_metrics
from src.config.vwap_campaign import PropFirmConstraintConfig, TimeWindow, VWAPVariantConfig
from src.data.cleaning import clean_ohlcv
from src.data.loader import load_ohlcv_file
from src.engine.execution_model import ExecutionModel
from src.engine.portfolio import build_equity_curve
from src.engine.vwap_backtester import InstrumentDetails, run_vwap_backtest
from src.strategy.vwap import build_vwap_signal_frame, prepare_vwap_feature_frame

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 200)


## 1) Parameters (edit here)


In [ ]:
ROOT = root

# ----------------------
# Data / split settings
# ----------------------
DATASET_PATH = Path(r"D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\processed\parquet\MNQ_c_0_1m_20260321_094501.parquet")
OUTPUT_DIR = Path(r"D:\Business\Trading\VSCODE\algo-trading-intraday-research\data\exports\vwap_full_smoke")
SYMBOL = "MNQ"
IS_FRACTION = 0.7
SESSION_START = "09:30:00"
SESSION_END = "16:00:00"
VWAP_PRICE_MODE = "typical"
ROLLING_WINDOW_DAYS = 20

# ----------------------
# Selected strategy settings
# ----------------------
SELECTED_VARIANT_NAME = "vwap_pullback_continuation"
VARIANT_FAMILY = "prop_variant"
MODE = "discrete"
EXECUTION_PROFILE = "repo_realistic"
INITIAL_CAPITAL_USD = 50000.0
QUANTITY_MODE = "fixed_quantity"
FIXED_QUANTITY = 1
TIME_WINDOWS = ()
SLOPE_LOOKBACK = 5
SLOPE_THRESHOLD = 0.0
ATR_PERIOD = 14
ATR_BUFFER = 0.3
COMPRESSION_LENGTH = 3
PULLBACK_LOOKBACK = 8
MAX_TRADES_PER_DAY = 3
MAX_LOSSES_PER_DAY = None
DAILY_STOP_THRESHOLD_USD = None
CONSECUTIVE_LOSSES_THRESHOLD = None
DELEVERAGE_AFTER_LOSING_STREAK = 1.0
RISK_PER_TRADE_PCT = None
EXIT_ON_VWAP_RECROSS = True
USE_PARTIAL_EXIT = False
PARTIAL_EXIT_R_MULTIPLE = 1.0
KEEP_RUNNER_UNTIL_CLOSE = True
VARIANT_NOTES = 'Trend continuation after a contained pullback inside a valid VWAP regime.'

# ----------------------
# Instrument / execution settings
# ----------------------
ASSET_CLASS = "futures"
TICK_SIZE = 0.25
TICK_VALUE_USD = 0.5
POINT_VALUE_USD = 2.0
COMMISSION_PER_SIDE_USD = 1.25
SLIPPAGE_TICKS = 1

# ----------------------
# Prop-style evaluation settings
# ----------------------
PROP_ACCOUNT_SIZE_USD = 50000.0
PROFIT_TARGET_PCT = 0.06
DAILY_LOSS_LIMIT_USD = 1000.0
TRAILING_DRAWDOWN_LIMIT_USD = 2000.0
CONSECUTIVE_RED_DAYS_THRESHOLD = 3
TRADING_DAYS_PER_MONTH = 21.0

# ----------------------
# Benchmark / notebook display settings
# ----------------------
BENCHMARK_LABEL = "Buy&Hold"
BENCHMARK_PRICE_COLUMN = "close"
BENCHMARK_INITIAL_CAPITAL_USD = INITIAL_CAPITAL_USD
RANKING_ROWS = 10
PLOT_TEMPLATE = "plotly_dark"
PLOT_WIDTH = 1800

print("DATASET_PATH =", DATASET_PATH)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("SELECTED_VARIANT_NAME =", SELECTED_VARIANT_NAME)
print("MODE =", MODE)
print("INITIAL_CAPITAL_USD =", INITIAL_CAPITAL_USD)
print("RISK_PER_TRADE_PCT =", RISK_PER_TRADE_PCT)
print("COMMISSION_PER_SIDE_USD =", COMMISSION_PER_SIDE_USD)
print("SLIPPAGE_TICKS =", SLIPPAGE_TICKS)


## 2) Full parameter snapshot


In [ ]:
def _format_windows(windows: tuple[tuple[str, str], ...]) -> str:
    if not windows:
        return "[]"
    return " | ".join([f"{start}->{end}" for start, end in windows])


parameter_snapshot = pd.DataFrame(
    [
        {"group": "data", "parameter": "DATASET_PATH", "value": str(DATASET_PATH)},
        {"group": "data", "parameter": "SYMBOL", "value": SYMBOL},
        {"group": "data", "parameter": "SESSION_START", "value": SESSION_START},
        {"group": "data", "parameter": "SESSION_END", "value": SESSION_END},
        {"group": "data", "parameter": "VWAP_PRICE_MODE", "value": VWAP_PRICE_MODE},
        {"group": "data", "parameter": "IS_FRACTION", "value": IS_FRACTION},
        {"group": "data", "parameter": "ROLLING_WINDOW_DAYS", "value": ROLLING_WINDOW_DAYS},
        {"group": "variant", "parameter": "SELECTED_VARIANT_NAME", "value": SELECTED_VARIANT_NAME},
        {"group": "variant", "parameter": "VARIANT_FAMILY", "value": VARIANT_FAMILY},
        {"group": "variant", "parameter": "MODE", "value": MODE},
        {"group": "variant", "parameter": "EXECUTION_PROFILE", "value": EXECUTION_PROFILE},
        {"group": "variant", "parameter": "INITIAL_CAPITAL_USD", "value": INITIAL_CAPITAL_USD},
        {"group": "variant", "parameter": "QUANTITY_MODE", "value": QUANTITY_MODE},
        {"group": "variant", "parameter": "FIXED_QUANTITY", "value": FIXED_QUANTITY},
        {"group": "variant", "parameter": "TIME_WINDOWS", "value": _format_windows(TIME_WINDOWS)},
        {"group": "variant", "parameter": "SLOPE_LOOKBACK", "value": SLOPE_LOOKBACK},
        {"group": "variant", "parameter": "SLOPE_THRESHOLD", "value": SLOPE_THRESHOLD},
        {"group": "variant", "parameter": "ATR_PERIOD", "value": ATR_PERIOD},
        {"group": "variant", "parameter": "ATR_BUFFER", "value": ATR_BUFFER},
        {"group": "variant", "parameter": "COMPRESSION_LENGTH", "value": COMPRESSION_LENGTH},
        {"group": "variant", "parameter": "PULLBACK_LOOKBACK", "value": PULLBACK_LOOKBACK},
        {"group": "variant", "parameter": "MAX_TRADES_PER_DAY", "value": MAX_TRADES_PER_DAY},
        {"group": "variant", "parameter": "MAX_LOSSES_PER_DAY", "value": MAX_LOSSES_PER_DAY},
        {"group": "variant", "parameter": "DAILY_STOP_THRESHOLD_USD", "value": DAILY_STOP_THRESHOLD_USD},
        {"group": "variant", "parameter": "CONSECUTIVE_LOSSES_THRESHOLD", "value": CONSECUTIVE_LOSSES_THRESHOLD},
        {"group": "variant", "parameter": "DELEVERAGE_AFTER_LOSING_STREAK", "value": DELEVERAGE_AFTER_LOSING_STREAK},
        {"group": "variant", "parameter": "RISK_PER_TRADE_PCT", "value": RISK_PER_TRADE_PCT},
        {"group": "variant", "parameter": "EXIT_ON_VWAP_RECROSS", "value": EXIT_ON_VWAP_RECROSS},
        {"group": "variant", "parameter": "USE_PARTIAL_EXIT", "value": USE_PARTIAL_EXIT},
        {"group": "variant", "parameter": "PARTIAL_EXIT_R_MULTIPLE", "value": PARTIAL_EXIT_R_MULTIPLE},
        {"group": "variant", "parameter": "KEEP_RUNNER_UNTIL_CLOSE", "value": KEEP_RUNNER_UNTIL_CLOSE},
        {"group": "variant", "parameter": "VARIANT_NOTES", "value": VARIANT_NOTES},
        {"group": "execution", "parameter": "ASSET_CLASS", "value": ASSET_CLASS},
        {"group": "execution", "parameter": "TICK_SIZE", "value": TICK_SIZE},
        {"group": "execution", "parameter": "TICK_VALUE_USD", "value": TICK_VALUE_USD},
        {"group": "execution", "parameter": "POINT_VALUE_USD", "value": POINT_VALUE_USD},
        {"group": "execution", "parameter": "COMMISSION_PER_SIDE_USD", "value": COMMISSION_PER_SIDE_USD},
        {"group": "execution", "parameter": "SLIPPAGE_TICKS", "value": SLIPPAGE_TICKS},
        {"group": "prop", "parameter": "PROP_ACCOUNT_SIZE_USD", "value": PROP_ACCOUNT_SIZE_USD},
        {"group": "prop", "parameter": "PROFIT_TARGET_PCT", "value": PROFIT_TARGET_PCT},
        {"group": "prop", "parameter": "DAILY_LOSS_LIMIT_USD", "value": DAILY_LOSS_LIMIT_USD},
        {"group": "prop", "parameter": "TRAILING_DRAWDOWN_LIMIT_USD", "value": TRAILING_DRAWDOWN_LIMIT_USD},
        {"group": "prop", "parameter": "CONSECUTIVE_RED_DAYS_THRESHOLD", "value": CONSECUTIVE_RED_DAYS_THRESHOLD},
        {"group": "prop", "parameter": "TRADING_DAYS_PER_MONTH", "value": TRADING_DAYS_PER_MONTH},
        {"group": "benchmark", "parameter": "BENCHMARK_LABEL", "value": BENCHMARK_LABEL},
        {"group": "benchmark", "parameter": "BENCHMARK_PRICE_COLUMN", "value": BENCHMARK_PRICE_COLUMN},
        {"group": "benchmark", "parameter": "BENCHMARK_INITIAL_CAPITAL_USD", "value": BENCHMARK_INITIAL_CAPITAL_USD},
    ]
)

display(parameter_snapshot)
display(Markdown(
    "### Notes\n"
    "- `USE_PARTIAL_EXIT`, `PARTIAL_EXIT_R_MULTIPLE` and `KEEP_RUNNER_UNTIL_CLOSE` stay explicit here for completeness.\n"
    "- In the current engine, the live logic mainly consumes the parameters that drive entry, stop, sizing, execution costs, prop limits and metrics."
))


## 3) Build data, signals, backtest, and benchmark


In [ ]:
def _split_sessions(all_sessions: list, is_fraction: float) -> tuple[list, list]:
    if len(all_sessions) < 2:
        raise ValueError("Not enough sessions to perform an IS/OOS split.")
    split_idx = int(len(all_sessions) * float(is_fraction))
    split_idx = max(1, min(len(all_sessions) - 1, split_idx))
    return all_sessions[:split_idx], all_sessions[split_idx:]


def _subset_frame_by_sessions(frame: pd.DataFrame, sessions: list) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    session_set = set(pd.to_datetime(pd.Index(sessions)).date)
    out = frame.copy()
    out_dates = pd.to_datetime(out["session_date"]).dt.date
    return out.loc[out_dates.isin(session_set)].copy().reset_index(drop=True)


def _subset_curve_to_sessions(curve: pd.DataFrame, sessions: list) -> pd.DataFrame:
    if curve.empty:
        return curve.copy()
    session_set = set(pd.to_datetime(pd.Index(sessions)).date)
    out = curve.copy()
    curve_dates = pd.to_datetime(out["timestamp"], errors="coerce").dt.date
    return out.loc[curve_dates.isin(session_set)].copy().reset_index(drop=True)


def _build_buy_hold_curve(frame: pd.DataFrame, initial_capital: float, price_column: str = "close") -> pd.DataFrame:
    bench_src = frame[["timestamp", price_column, "session_date"]].copy()
    bench_src["timestamp"] = pd.to_datetime(bench_src["timestamp"], utc=True, errors="coerce").dt.tz_convert(None)
    bench_src[price_column] = pd.to_numeric(bench_src[price_column], errors="coerce")
    bench_src = bench_src.dropna(subset=["timestamp", price_column])
    daily_close = bench_src.groupby("session_date")[price_column].last().dropna()
    if daily_close.empty:
        return pd.DataFrame(columns=["timestamp", "equity", "drawdown", "drawdown_pct"])

    out = pd.DataFrame(
        {
            "timestamp": pd.to_datetime(daily_close.index),
            "equity": float(initial_capital) * (daily_close / float(daily_close.iloc[0])),
        }
    ).sort_values("timestamp")
    out["peak"] = out["equity"].cummax()
    out["drawdown"] = out["equity"] - out["peak"]
    return normalize_curve(out.drop(columns=["peak"]).reset_index(drop=True))


VARIANT = VWAPVariantConfig(
    name=SELECTED_VARIANT_NAME,
    family=VARIANT_FAMILY,
    mode=MODE,
    execution_profile=EXECUTION_PROFILE,
    initial_capital_usd=INITIAL_CAPITAL_USD,
    quantity_mode=QUANTITY_MODE,
    fixed_quantity=FIXED_QUANTITY,
    time_windows=tuple(TimeWindow(start, end) for start, end in TIME_WINDOWS),
    slope_lookback=SLOPE_LOOKBACK,
    slope_threshold=SLOPE_THRESHOLD,
    atr_period=ATR_PERIOD,
    atr_buffer=ATR_BUFFER,
    compression_length=COMPRESSION_LENGTH,
    pullback_lookback=PULLBACK_LOOKBACK,
    max_trades_per_day=MAX_TRADES_PER_DAY,
    max_losses_per_day=MAX_LOSSES_PER_DAY,
    daily_stop_threshold_usd=DAILY_STOP_THRESHOLD_USD,
    consecutive_losses_threshold=CONSECUTIVE_LOSSES_THRESHOLD,
    deleverage_after_losing_streak=DELEVERAGE_AFTER_LOSING_STREAK,
    risk_per_trade_pct=RISK_PER_TRADE_PCT,
    exit_on_vwap_recross=EXIT_ON_VWAP_RECROSS,
    use_partial_exit=USE_PARTIAL_EXIT,
    partial_exit_r_multiple=PARTIAL_EXIT_R_MULTIPLE,
    keep_runner_until_close=KEEP_RUNNER_UNTIL_CLOSE,
    notes=VARIANT_NOTES,
)

INSTRUMENT = InstrumentDetails(
    symbol=SYMBOL,
    asset_class=ASSET_CLASS,
    tick_size=TICK_SIZE,
    tick_value_usd=TICK_VALUE_USD,
    point_value_usd=POINT_VALUE_USD,
    commission_per_side_usd=COMMISSION_PER_SIDE_USD,
    slippage_ticks=SLIPPAGE_TICKS,
)

EXECUTION_MODEL = ExecutionModel(
    commission_per_side_usd=COMMISSION_PER_SIDE_USD,
    slippage_ticks=SLIPPAGE_TICKS,
    tick_size=TICK_SIZE,
)

PROP_CONSTRAINTS = PropFirmConstraintConfig(
    name="notebook_prop_reference",
    account_size_usd=PROP_ACCOUNT_SIZE_USD,
    profit_target_pct=PROFIT_TARGET_PCT,
    daily_loss_limit_usd=DAILY_LOSS_LIMIT_USD,
    trailing_drawdown_limit_usd=TRAILING_DRAWDOWN_LIMIT_USD,
    consecutive_red_days_threshold=CONSECUTIVE_RED_DAYS_THRESHOLD,
    trading_days_per_month=TRADING_DAYS_PER_MONTH,
)

raw = load_ohlcv_file(DATASET_PATH)
clean = clean_ohlcv(raw)
feature_frame = prepare_vwap_feature_frame(
    clean,
    session_start=SESSION_START,
    session_end=SESSION_END,
    atr_windows=(ATR_PERIOD,),
    vwap_price_mode=VWAP_PRICE_MODE,
)
signal_df = build_vwap_signal_frame(feature_frame, VARIANT)
result = run_vwap_backtest(signal_df, VARIANT, EXECUTION_MODEL, INSTRUMENT)

all_sessions = sorted(pd.to_datetime(feature_frame["session_date"]).dt.date.unique())
is_sessions, oos_sessions = _split_sessions(all_sessions, IS_FRACTION)

metrics_overall, rolling_table = compute_extended_vwap_metrics(
    trades=result.trades,
    daily_results=result.daily_results,
    bar_results=result.bar_results,
    signal_df=signal_df,
    initial_capital=INITIAL_CAPITAL_USD,
    prop_constraints=PROP_CONSTRAINTS,
    rolling_window_days=ROLLING_WINDOW_DAYS,
)
metrics_is, _ = compute_extended_vwap_metrics(
    trades=_subset_frame_by_sessions(result.trades, is_sessions),
    daily_results=_subset_frame_by_sessions(result.daily_results, is_sessions),
    bar_results=_subset_frame_by_sessions(result.bar_results, is_sessions),
    signal_df=_subset_frame_by_sessions(signal_df, is_sessions),
    initial_capital=INITIAL_CAPITAL_USD,
    prop_constraints=PROP_CONSTRAINTS,
    rolling_window_days=ROLLING_WINDOW_DAYS,
)
metrics_oos, _ = compute_extended_vwap_metrics(
    trades=_subset_frame_by_sessions(result.trades, oos_sessions),
    daily_results=_subset_frame_by_sessions(result.daily_results, oos_sessions),
    bar_results=_subset_frame_by_sessions(result.bar_results, oos_sessions),
    signal_df=_subset_frame_by_sessions(signal_df, oos_sessions),
    initial_capital=INITIAL_CAPITAL_USD,
    prop_constraints=PROP_CONSTRAINTS,
    rolling_window_days=ROLLING_WINDOW_DAYS,
)

selected_curve = normalize_curve(build_equity_curve(result.trades, initial_capital=INITIAL_CAPITAL_USD))
selected_curve_oos = normalize_curve(
    build_equity_curve(
        _subset_frame_by_sessions(result.trades, oos_sessions),
        initial_capital=INITIAL_CAPITAL_USD,
    )
)

feature_oos = feature_frame.loc[pd.to_datetime(feature_frame["session_date"]).dt.date.isin(set(oos_sessions))].copy()
benchmark_curve = _build_buy_hold_curve(
    feature_frame,
    BENCHMARK_INITIAL_CAPITAL_USD,
    price_column=BENCHMARK_PRICE_COLUMN,
)
benchmark_curve_is = _subset_curve_to_sessions(benchmark_curve, is_sessions)
benchmark_curve_oos = _build_buy_hold_curve(
    feature_oos,
    BENCHMARK_INITIAL_CAPITAL_USD,
    price_column=BENCHMARK_PRICE_COLUMN,
)

ranking_path = OUTPUT_DIR / "summary" / "prop_variant_ranking.csv"
comparative_path = OUTPUT_DIR / "summary" / "comparative_metrics.csv"
ranking_df = pd.read_csv(ranking_path) if ranking_path.exists() else pd.DataFrame()
comparative_df = pd.read_csv(comparative_path) if comparative_path.exists() else pd.DataFrame()

print("Sessions total / IS / OOS =", len(all_sessions), "/", len(is_sessions), "/", len(oos_sessions))
print("Trades overall / OOS =", len(result.trades), "/", len(_subset_frame_by_sessions(result.trades, oos_sessions)))
print("Net PnL overall =", float(metrics_overall.get("net_pnl", 0.0)))
print("Sharpe overall / OOS =", float(metrics_overall.get("sharpe_ratio", 0.0)), "/", float(metrics_oos.get("sharpe_ratio", 0.0)))


## 4) Selected variant vs Buy and Hold


In [ ]:
selected_ret = curve_total_return_pct(selected_curve, INITIAL_CAPITAL_USD)
selected_dd_pct = curve_max_drawdown_pct(selected_curve)
selected_cagr_pct = curve_annualized_return(selected_curve, INITIAL_CAPITAL_USD) * 100.0
selected_vol_pct = curve_daily_vol(selected_curve) * 100.0
selected_overall_sh = float(metrics_overall.get("sharpe_ratio", 0.0))
selected_overall_pf = float(metrics_overall.get("profit_factor", 0.0))
selected_overall_exp = float(metrics_overall.get("expectancy_per_trade", 0.0))

selected_oos_ret = curve_total_return_pct(selected_curve_oos, INITIAL_CAPITAL_USD)
selected_oos_dd_pct = curve_max_drawdown_pct(selected_curve_oos)
selected_oos_sh = float(metrics_oos.get("sharpe_ratio", 0.0))
selected_oos_pf = float(metrics_oos.get("profit_factor", 0.0))
selected_oos_exp = float(metrics_oos.get("expectancy_per_trade", 0.0))

bench_ret = curve_total_return_pct(benchmark_curve, BENCHMARK_INITIAL_CAPITAL_USD)
bench_dd_pct = curve_max_drawdown_pct(benchmark_curve)
bench_sh = curve_daily_sharpe(benchmark_curve)
bench_cagr_pct = curve_annualized_return(benchmark_curve, BENCHMARK_INITIAL_CAPITAL_USD) * 100.0
bench_vol_pct = curve_daily_vol(benchmark_curve) * 100.0
bench_is_sh = curve_daily_sharpe(benchmark_curve_is)
bench_oos_sh = curve_daily_sharpe(benchmark_curve_oos)

fig = make_subplots(
    rows=2,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.08,
    row_heights=[0.70, 0.30],
    subplot_titles=("Equity Curve (USD)", "Drawdown (%)"),
)

fig.add_trace(
    go.Scatter(
        x=selected_curve["timestamp"],
        y=selected_curve["equity"],
        mode="lines",
        name=f"{SELECTED_VARIANT_NAME} Full Sample | Ret {selected_ret:.1f}% | Sharpe {selected_overall_sh:.2f} | PF {selected_overall_pf:.2f} | Exp/trade {selected_overall_exp:.1f}",
        line=dict(width=3.0, color="#22c55e"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=selected_curve["timestamp"],
        y=selected_curve["drawdown_pct"],
        mode="lines",
        name="DD Selected Variant",
        showlegend=False,
        line=dict(width=1.7, color="#22c55e", dash="dot"),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=selected_curve_oos["timestamp"],
        y=selected_curve_oos["equity"],
        mode="lines",
        name=f"{SELECTED_VARIANT_NAME} OOS Only | Ret {selected_oos_ret:.1f}% | Sharpe {selected_oos_sh:.2f} | PF {selected_oos_pf:.2f} | Exp/trade {selected_oos_exp:.1f}",
        line=dict(width=2.4, color="#f59e0b"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=selected_curve_oos["timestamp"],
        y=selected_curve_oos["drawdown_pct"],
        mode="lines",
        name="DD Selected Variant OOS",
        showlegend=False,
        line=dict(width=1.5, color="#f59e0b", dash="dot"),
    ),
    row=2,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=benchmark_curve["timestamp"],
        y=benchmark_curve["equity"],
        mode="lines",
        name=f"{BENCHMARK_LABEL} | Ret {bench_ret:.1f}% | MaxDD {bench_dd_pct:.1f}%",
        line=dict(width=2.6, color="#38bdf8"),
    ),
    row=1,
    col=1,
)
fig.add_trace(
    go.Scatter(
        x=benchmark_curve["timestamp"],
        y=benchmark_curve["drawdown_pct"],
        mode="lines",
        name="DD Buy&Hold",
        showlegend=False,
        line=dict(width=1.5, color="#38bdf8", dash="dot"),
    ),
    row=2,
    col=1,
)

fig.update_layout(
    template=PLOT_TEMPLATE,
    width=int(PLOT_WIDTH),
    height=900,
    title=f"VWAP Selected Variant vs {BENCHMARK_LABEL} | {SYMBOL}",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0.0),
)
fig.update_yaxes(title_text="Equity (USD)", row=1, col=1)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=1)
fig.show()

display(Markdown(build_scope_readout_markdown(
    full_curve=selected_curve,
    oos_curve=selected_curve_oos,
    initial_capital=INITIAL_CAPITAL_USD,
    full_label=f"{SELECTED_VARIANT_NAME} full-sample curve",
    oos_label=f"{SELECTED_VARIANT_NAME} OOS-only curve",
)))

display(Markdown(
    "### Strategy vs benchmark\n"
    f"- {format_curve_stats_line(SELECTED_VARIANT_NAME, selected_overall_sh, selected_ret, selected_cagr_pct, selected_vol_pct, selected_dd_pct, pf=selected_overall_pf, exp=selected_overall_exp)}\n"
    f"- {format_curve_stats_line(BENCHMARK_LABEL, bench_sh, bench_ret, bench_cagr_pct, bench_vol_pct, bench_dd_pct)}"
))


## 5) KPI table


In [ ]:
kpi = pd.DataFrame([
    {
        "model": SELECTED_VARIANT_NAME,
        "overall_sharpe": selected_overall_sh,
        "is_sharpe": float(metrics_is.get("sharpe_ratio", 0.0)),
        "oos_sharpe": selected_oos_sh,
        "overall_profit_factor": selected_overall_pf,
        "oos_profit_factor": selected_oos_pf,
        "overall_expectancy_per_trade": selected_overall_exp,
        "oos_expectancy_per_trade": selected_oos_exp,
        "total_return_pct": selected_ret,
        "cagr_pct": selected_cagr_pct,
        "vol_pct": selected_vol_pct,
        "max_drawdown_pct": selected_dd_pct,
        "overall_net_pnl": float(metrics_overall.get("net_pnl", 0.0)),
        "oos_net_pnl": float(metrics_oos.get("net_pnl", 0.0)),
        "overall_n_trades": int(metrics_overall.get("n_trades", 0)),
        "oos_n_trades": int(metrics_oos.get("n_trades", 0)),
        "days_to_target_pct": metrics_overall.get("days_to_target_pct"),
        "daily_loss_limit_breach_freq": float(metrics_overall.get("daily_loss_limit_breach_freq", 0.0)),
        "trailing_drawdown_breach_freq": float(metrics_overall.get("trailing_drawdown_breach_freq", 0.0)),
        "profit_to_drawdown_ratio": float(metrics_overall.get("profit_to_drawdown_ratio", 0.0)),
    },
    {
        "model": BENCHMARK_LABEL,
        "overall_sharpe": bench_sh,
        "is_sharpe": bench_is_sh,
        "oos_sharpe": bench_oos_sh,
        "overall_profit_factor": np.nan,
        "oos_profit_factor": np.nan,
        "overall_expectancy_per_trade": np.nan,
        "oos_expectancy_per_trade": np.nan,
        "total_return_pct": bench_ret,
        "cagr_pct": bench_cagr_pct,
        "vol_pct": bench_vol_pct,
        "max_drawdown_pct": bench_dd_pct,
        "overall_net_pnl": float(benchmark_curve["equity"].iloc[-1] - BENCHMARK_INITIAL_CAPITAL_USD) if not benchmark_curve.empty else np.nan,
        "oos_net_pnl": float(benchmark_curve_oos["equity"].iloc[-1] - BENCHMARK_INITIAL_CAPITAL_USD) if not benchmark_curve_oos.empty else np.nan,
        "overall_n_trades": np.nan,
        "oos_n_trades": np.nan,
        "days_to_target_pct": np.nan,
        "daily_loss_limit_breach_freq": np.nan,
        "trailing_drawdown_breach_freq": np.nan,
        "profit_to_drawdown_ratio": np.nan,
    },
])
display(kpi)


## 6) Export snapshot


In [ ]:
if ranking_df.empty:
    print("No export ranking snapshot found in", OUTPUT_DIR / "summary")
else:
    display(Markdown(
        "### Export ranking snapshot\n"
        "- This table comes from the saved campaign export in `OUTPUT_DIR`.\n"
        "- If you manually edit the parameters above, this snapshot does not auto-refresh."
    ))
    display(
        ranking_df[
            [
                "name",
                "selection_score",
                "oos_net_pnl",
                "oos_sharpe_ratio",
                "oos_profit_factor",
                "oos_max_drawdown",
                "oos_daily_loss_limit_breach_freq",
                "oos_profit_to_drawdown_ratio",
            ]
        ].head(int(RANKING_ROWS))
    )
